# Lab 01: Phân Loại Ảnh FashionMNIST Với PyTorch
> **Kiến trúc:** Clean Architecture & Modular `src/` (Chạy mượt mà trên cả máy cục bộ và Kaggle GPU)  
> **Mô hình:** `FashionMLP` thuần túy PyTorch với 3 loại layer cơ bản (`Flatten`, `Linear`, `ReLU`)  
> **Quy trình:** Core EDA & FE -> Pipeline Dữ Liệu -> Huấn Luyện FashionMLP -> Optuna HPO & Benchmark 3 Optimizer -> ROC-AUC & Decision Threshold Scan


## 1. Thiết Lập Môi Trường & Nạp Modules Từ `src/`


In [ ]:
import sys, os, shutil
from pathlib import Path
import torch
import matplotlib.pyplot as plt

# 1. Thiet lap thu muc src (ho tro ca Kaggle input va Local)
root = list(Path("/kaggle/input").glob("**/src"))
if root:
    src_parent = str(root[0].parent)
    if src_parent not in sys.path:
        sys.path.insert(0, src_parent)
else:
    sys.path.insert(0, str(Path("..").resolve()))

from src import (
    DATA_DIR, EDA_DIR, OUTPUTS_DIR, CHECKPOINTS_DIR,
    DEVICE, RANDOM_SEED, CLASS_NAMES, VAL_SPLIT,
    set_seed, get_dataloaders, build_model, count_parameters,
    train_model, evaluate, sanity_overfit_check,
    run_eda_and_fe_pipeline, verify_pipeline_handoff,
    plot_loss_and_acc_curves, plot_confusion_matrix, plot_prediction_grid,
    run_optuna_optimizer_study, compare_optimizers_wallclock,
    plot_roc_curves, run_threshold_scan, save_checkpoint,
)

set_seed(RANDOM_SEED)
print(f"[DEVICE] Thiết bị tính toán: {DEVICE} | [SEED] Hạt ngẫu nhiên: {RANDOM_SEED} | [CLASSES] Số lớp phân loại: {len(CLASS_NAMES)}")


## 2. Khám Phá Dữ Liệu Chuyên Sâu (Core EDA) & Khóa Cấu Hình Tiền Xử Lý (Feature Engineering)
> Thực hiện các bước Core EDA & FE theo đúng tài liệu kỹ thuật `docs/eda.md`, `docs/eda-image.md`, `docs/feature-image.md` và `docs/dl_data.md`:  
> 1. **Structural Overview:** Kích thước, kiểu dữ liệu uint8, dải điểm ảnh [0, 255] và dung lượng RAM.  
> 2. **Integrity Check:** Quét tính toàn vẹn (không có NaN, đúng 10 lớp [0, 9], trùng lặp nội bộ Train/Test và 0 mẫu rò rỉ chéo Train vs Test).  
> 3. **Stratified Split & Normalization Lock:** Tách phân vùng ngẫu nhiên có phân tầng (Stratified Split), đo đạc Mean/Std trên đúng tập Train chống rò rỉ dữ liệu (Anti-Leakage), và lưu file cấu hình `preprocessing_config.json`.


In [ ]:
eda_results = run_eda_and_fe_pipeline(
    data_dir=DATA_DIR,
    eda_dir=EDA_DIR,
    val_split=VAL_SPLIT,
    seed=RANDOM_SEED,
    class_names=CLASS_NAMES,
    verbose=True,
)


## 3. Nạp Dữ Liệu PyTorch DataLoaders & Cổng Kiểm Tra Bàn Giao (Guard Assert Gate)
> Pipeline nạp dữ liệu sạch: Áp dụng phân chia Stratified Split đã khóa, biến đổi chuẩn hóa `ToTensor` -> `Normalize(mean, std)` độc lập chống rò rỉ dữ liệu, và xác thực đủ 4 tiêu chí bàn giao trước khi huấn luyện.


In [ ]:
train_loader, val_loader, test_loader, _ = get_dataloaders(
    batch_size=128,
    augment=False,
    num_workers=0,
)
verify_pipeline_handoff(train_loader, val_loader, eda_dir=EDA_DIR)
print(f"[NẠP DỮ LIỆU THÀNH CÔNG] Tập Train: {len(train_loader)} batches | Tập Val: {len(val_loader)} batches | Tập Test: {len(test_loader)} batches")


## 4. Kiến Trúc Mô Hình FashionMLP & Kiểm Tra Sanity Overfit 1 Batch
> **Kiến trúc:** `Flatten` -> `Linear(784, 256)` -> `ReLU` -> `Linear(256, 128)` -> `ReLU` -> `Linear(128, 10)`.  
> **Kiểm tra Sanity (TDD Gate):** Đảm bảo mô hình có khả năng học và kéo loss trên 1 mini-batch nhỏ về sát 0 trước khi huấn luyện diện rộng.


In [ ]:
model = build_model("mlp", num_classes=10).to(DEVICE)
params = count_parameters(model)
print(model)
print(f"[THAM SỐ MÔ HÌNH] Tổng tham số: {params['total']:,} | Tham số cần huấn luyện: {params['trainable']:,}")

sample_images, sample_targets = next(iter(train_loader))
sanity_pass, s_loss, s_acc = sanity_overfit_check(
    model=model,
    sample_images=sample_images[:16],
    sample_targets=sample_targets[:16],
    num_steps=120,
    target_loss=0.05,
    device=DEVICE,
)
print(f"[SANITY PASS] Kiểm tra Sanity thành công! Loss = {s_loss:.4f} | Accuracy = {s_acc:.1f}%")


## 5. Huấn Luyện Mô Hình FashionMLP (10 Epochs)


In [ ]:
# Khởi tạo lại mô hình để xóa trọng số của bước kiểm tra sanity
model = build_model("mlp", num_classes=10).to(DEVICE)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history, best_val_acc = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=10,
    device=DEVICE,
    save_path=CHECKPOINTS_DIR / "fashion_mlp_best.pt",
)
print(f"[HUẤN LUYỆN HOÀN TẤT] Độ chính xác tập Validation cao nhất: {best_val_acc:.2f}%")


## 6. Đánh Giá Hiệu Năng, Ma Trận Nhầm Lẫn & Lưới 16 Ảnh Dự Đoán


In [ ]:
fig_curves = plot_loss_and_acc_curves(
    history,
    title="FashionMLP: Đường Cong Học Tập (Loss & Accuracy)",
    save_path=OUTPUTS_DIR / "fashion_mlp_learning_curves.png",
)
plt.show()

val_loss, val_acc, all_preds, all_targets = evaluate(model, val_loader, criterion, DEVICE)

fig_cm = plot_confusion_matrix(
    all_targets, all_preds, CLASS_NAMES,
    title="FashionMLP: Ma Trận Nhầm Lẫn (Confusion Matrix)",
    save_path=OUTPUTS_DIR / "confusion_matrix.png",
)
plt.show()

fig_grid = plot_prediction_grid(
    model, val_loader, CLASS_NAMES, DEVICE, num_images=16,
    save_path=OUTPUTS_DIR / "predictions_sample.png",
)
plt.show()


## 7. So Sánh 3 Optimizer: Cùng Learning Rate vs. Tối Ưu Bằng Optuna
> **Kịch bản 1:** So sánh công bằng bước đầu khi cả 3 thuật toán (SGD, RMSprop, Adam) dùng **chung một Learning Rate `1e-3`**.  
> **Kịch bản 2:** Dùng **Optuna (Bayesian TPE + MedianPruner)** tự động tìm kiếm siêu tham số tối ưu riêng cho từng thuật toán, sau đó so tài đỉnh-với-đỉnh theo thời gian thực (Wall-clock seconds trên GPU) để chọn ra `winner_model`.


In [ ]:
print("=== KỊCH BẢN 1: SO SÁNH 3 OPTIMIZER VỚI CÙNG LEARNING RATE (lr = 1e-3) ===")
fixed_configs = {
    "SGD": {"lr": 1e-3, "momentum": 0.9},
    "RMSprop": {"lr": 1e-3},
    "Adam": {"lr": 1e-3},
}
fixed_df, _, _, fig_fixed = compare_optimizers_wallclock(
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer_configs=fixed_configs,
    num_epochs=8,
    target_accuracy=87.0,
    device=DEVICE,
    seed=RANDOM_SEED,
    save_fig_path=OUTPUTS_DIR / "optimizer_fixed_lr_comparison.png",
)
print(fixed_df.to_string(index=False))
plt.show()

print("\n=== KỊCH BẢN 2: DÒ TÌM SIÊU THAM SỐ VỚI OPTUNA (BAYESIAN TPE) ===")
best_params, trials_df = run_optuna_optimizer_study(
    train_loader=train_loader,
    val_loader=val_loader,
    num_trials=8,
    epochs_per_trial=2,
    device=DEVICE,
    seed=RANDOM_SEED,
)
print("\n--- BẢNG LỊCH SỬ CÁC TRIAL OPTUNA ---")
print(trials_df.to_string(index=False))

print("\n=== SO SÁNH ĐỐI ĐẦU ĐỈNH-VỚI-ĐỈNH THEO THỜI GIAN THỰC ===")
benchmark_df, winner_model, winner_name, fig_bench = compare_optimizers_wallclock(
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer_configs=best_params,
    num_epochs=10,
    target_accuracy=87.0,
    device=DEVICE,
    seed=RANDOM_SEED,
    save_fig_path=OUTPUTS_DIR / "optimizer_wallclock_comparison.png",
)
print(benchmark_df.to_string(index=False))
print(f"\n[WINNER OPTIMIZER] Thuật toán tối ưu chiến thắng: {winner_name}")
plt.show()


## 8. Khảo Sát Đường Cong ROC & Điểm ROC-AUC Đa Lớp (One-vs-Rest)
> Đánh giá năng lực phân biệt (Discrimination Power) của `winner_model` trên toàn bộ 10 lớp áo quần độc lập với ngưỡng phân loại.


In [ ]:
roc_results, fig_roc = plot_roc_curves(
    model=winner_model,
    dataloader=val_loader,
    class_names=CLASS_NAMES,
    device=DEVICE,
    save_fig_path=OUTPUTS_DIR / "multiclass_roc_curves.png",
)
plt.show()


## 9. Tối Ưu Hóa Ngưỡng Quyết Định Phân Loại (Decision Threshold Scan) & Đóng Gói Output
> Quét 100 ngưỡng xác suất trên `winner_model` cho lớp trọng tâm, vẽ 4 đường chỉ số (Precision, Recall, F1-Score, Accuracy) kèm vạch đỏ đánh dấu ngưỡng tối ưu t*, và in bảng theo dõi bước 0.05.


In [ ]:
opt_t, opt_f1, tracking_df, fig_thresh = run_threshold_scan(
    model=winner_model,
    dataloader=val_loader,
    positive_class_idx=0,
    device=DEVICE,
    num_thresholds=100,
    step_display=0.05,
    title=f"Decision Threshold Scan ({winner_name}): Lớp '{CLASS_NAMES[0]}' (T-shirt/top)",
    save_fig_path=OUTPUTS_DIR / "decision_threshold_scan.png",
)
plt.show()

# Lưu checkpoint mô hình chiến thắng cuối cùng đúng chữ ký (model, save_path, metadata)
save_checkpoint(
    model=winner_model,
    save_path=CHECKPOINTS_DIR / "fashion_mlp_final.pt",
    metadata={"winner_optimizer": winner_name, "optimal_threshold_class0": opt_t, "num_epochs": 10},
)

# Đóng gói toàn bộ artifacts thành 1 file zip tiện lợi để tải về
shutil.make_archive(str(BASE_DIR / "all_outputs"), 'zip', root_dir=str(OUTPUTS_DIR))
print("[HOÀN TẤT] Pipeline đã thực thi thành công! Đã tạo file 'all_outputs.zip' tại thư mục gốc để tải về trực tiếp!")
